# Prepare the data

In [ ]:
# sotify app uri: http://127.0.0.1:9090

import spotipy
import os
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv

load_dotenv()

client_id = os.getenv("SPOTIFY_CLIENT_ID")
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=client_id,
                                               client_secret=client_secret,
                                               redirect_uri="http://127.0.0.1:9090",
                                               scope="user-library-read"))



In [13]:
# Load the data
import pandas as pd
import numpy as np

data = pd.read_csv('../data/raw/dataset.csv', na_values=["", "NaN", "-", "null", "Null", "none", "None"])


# Drop NaNs and irrelevant columns, define features array and target variable
data = data.dropna()
data.drop_duplicates(inplace=True)

In [14]:
# Filter out all the samples that are copies of the original song
artist_med = data.groupby('artists')['popularity'].transform('median')
junk = (artist_med > 30) & (data['popularity'] < 10)

clean_data = data[~junk].copy()

clean_data["primary_artist"] = clean_data["artists"].str.split(";").str[0].str.strip()
g = clean_data.groupby(["primary_artist", "duration_ms"])
g.filter(lambda x: len(x) > 1).sort_values(["primary_artist", "duration_ms"])[
    ["track_name", "album_name", "popularity"]
].nunique()

track_name    15639
album_name    11603
popularity      100
dtype: int64

In [15]:
data[data['popularity'] == 0]

,track_id,spotify_track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
23,23,0BUuuEvNa5T4lMaewyiudB,Jason Mraz,Coffee Moment,93 Million Miles,0,216386,False,0.572,0.454,...,-10.286,1,0.0258,0.4770,0.000014,0.0974,0.515,140.182,4,acoustic
24,24,3Hn3LfhrQOaKihdCibJsTs,Jason Mraz,Human - Best Adult Pop Tunes,Unlonely,0,231266,False,0.796,0.667,...,-4.831,0,0.0392,0.3810,0.000000,0.2210,0.754,97.988,4,acoustic
26,26,5IfCZDRXZrqZSm8AwE44PG,Jason Mraz,Holly Jolly Christmas,Winter Wonderland,0,131760,False,0.620,0.309,...,-9.209,1,0.0495,0.7880,0.000000,0.1460,0.664,145.363,4,acoustic
27,27,0dzKBptH2P5j5a0MifBMwM,Jason Mraz,Feeling Good - Adult Pop Favorites,If It Kills Me,0,273653,False,0.633,0.429,...,-6.784,0,0.0381,0.0444,0.000000,0.1320,0.520,143.793,4,acoustic
28,28,5QAMZTM5cmLg3fHX9ZbTZi,Jason Mraz,Christmas Time,Winter Wonderland,0,131760,False,0.620,0.309,...,-9.209,1,0.0495,0.7880,0.000000,0.1460,0.664,145.363,4,acoustic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113046,113046,3W73vndiqDkaYwYjSMaSWu,Rend Collective,Santa's Christmas List,Ding Dong Merrily On High (The Celebration's S...,0,117226,False,0.696,0.715,...,-7.741,1,0.0314,0.3400,0.000000,0.1800,0.959,130.021,4,world-music
113047,113047,18FIFh4AGPC2zDLLFkJp4j,Kim Walker-Smith,Santa's Christmas List,Rudolph The Red-Nosed Reindeer,0,175426,False,0.581,0.436,...,-7.936,1,0.0443,0.2320,0.000000,0.0547,0.412,120.138,4,world-music
113048,113048,6xw4sP2mGqAtVYZkBojxcI,Kim Walker-Smith,Santa's Christmas List,I'll Be Home For Christmas,0,193653,False,0.413,0.437,...,-7.983,0,0.0327,0.0420,0.000036,0.1460,0.102,120.259,4,world-music
113049,113049,6E7Ix5jkd6uzfoxuvcI8Ww,Rend Collective;We The Kingdom,Santa's Christmas List,God Rest Ye Merry Gentlemen (Hallelujah),0,217120,False,0.607,0.884,...,-4.059,1,0.0489,0.0230,0.000000,0.2260,0.555,139.988,4,world-music


In [16]:
# Drop all the duplicate songs that have the same primary artists and duration
POP = "popularity"
ID  = "spotify_track_id"

clean_data["primary_artist"] = clean_data["artists"].str.split(";").str[0].str.strip()
clean_data = clean_data.sort_values(POP, ascending=False)        # keep="first" now == keep max popularity

clean_data = clean_data.drop_duplicates(["primary_artist", "duration_ms"], keep="first").reset_index(drop=True)
clean_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 81707 entries, 0 to 81706
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   track_id          81707 non-null  int64  
 1   spotify_track_id  81707 non-null  str    
 2   artists           81707 non-null  str    
 3   album_name        81707 non-null  str    
 4   track_name        81707 non-null  str    
 5   popularity        81707 non-null  int64  
 6   duration_ms       81707 non-null  int64  
 7   explicit          81707 non-null  bool   
 8   danceability      81707 non-null  float64
 9   energy            81707 non-null  float64
 10  key               81707 non-null  int64  
 11  loudness          81707 non-null  float64
 12  mode              81707 non-null  int64  
 13  speechiness       81707 non-null  float64
 14  acousticness      81707 non-null  float64
 15  instrumentalness  81707 non-null  float64
 16  liveness          81707 non-null  float64
 17  vale

In [17]:
# Remove rubbish samples with extreme duration and tempo values
MAX_MS  = 10 * 60 * 1000      # 600_000 ms
MIN_BPM = 40

before = len(clean_data)
junk = (clean_data["duration_ms"] >= MAX_MS) | (clean_data["tempo"] <= MIN_BPM) | (clean_data["duration_ms"] <= 60000)
clean_data = clean_data[~junk].reset_index(drop=True)
print(f"dropped {before - len(clean_data)} rows ({(before - len(clean_data)) / before:.1%})")

dropped 1407 rows (1.7%)


In [18]:
# Add new temporary feature - normalized song name
import re

ARTIST_COL, NAME_COL = "artists", "track_name"  # match your schema

QUALIFIER = (r"remaster(?:ed)?|remix|live|acoustic|deluxe|expanded|version|edit|"
             r"edition|mix|mono|stereo|radio|instrumental|demo|bonus|re-?recorded|"
             r"anniversary|single version")

def norm_name(name):
    s = str(name).lower().replace("\u2019", "'")                      # unify ’ and '
    s = re.sub(r"\s*[\(\[]?\s*(?:feat|ft|featuring)\.?\s.*?(?:[\)\]]|$)", " ", s)
    s = re.sub(rf"\s*[\(\[][^\)\]]*(?:{QUALIFIER})[^\)\]]*[\)\]]", " ", s)  # parens w/ qualifier only
    s = re.sub(rf"\s*-\s*[^-]*(?:{QUALIFIER}).*$", " ", s)             # trailing " - Live" etc.
    s = re.sub(r"[^a-z0-9 ]", "", s)                                  # drop remaining punctuation
    return re.sub(r"\s+", " ", s).strip()

clean_data["song_key"] = (clean_data[ARTIST_COL].str.split(";").str[0].str.strip().str.lower()
                  + " :: " + clean_data[NAME_COL].map(norm_name))

In [19]:
# Drop duplicates that have the same song_key
before = len(clean_data)
clean_data = clean_data.drop_duplicates("song_key", keep="first").drop(columns="song_key").reset_index(drop=True)
print(f"dropped {before - len(clean_data)} rows ({(before - len(clean_data)) / before:.1%})")

dropped 6638 rows (8.3%)


In [20]:
# explode to one row per (track, individual artist); original index preserved
df = clean_data[['artists', 'popularity']].copy()
df['artist'] = df['artists'].str.split(';')
ex = df.explode('artist')
ex['artist'] = ex['artist'].str.strip()

# per-row leave-one-out median within each artist
def loo_median(s):
    vals = s.to_numpy()
    n = vals.size
    if n == 1:                       # single track -> no "other" tracks
        return np.array([np.nan])
    return np.array([np.median(np.delete(vals, i)) for i in range(n)])

ex['artist_loo_med'] = ex.groupby('artist')['popularity'].transform(loo_median)

# collapse back to track level: max across the track's artists (skips NaN)
fame = ex.groupby(level=0)['artist_loo_med'].max()
clean_data['artist_fame_loo'] = fame

# tracks where every artist had only one song -> no signal -> neutral fill
clean_data['artist_fame_loo'] = clean_data['artist_fame_loo'].fillna(
    clean_data['popularity'].median()
)

In [21]:
before = len(clean_data)
junk = (clean_data['artist_fame_loo'] > 30) & (clean_data['popularity'] < 10)
clean_data = clean_data[~junk]
print(f"dropped {before - len(clean_data)} rows ({(before - len(clean_data)) / before:.1%})")

dropped 1627 rows (2.2%)


In [22]:
# Drop unnecessary columns
data = clean_data.copy()
clean_data = clean_data.drop(columns=["track_id", "spotify_track_id", "artists", "album_name", "track_name", "primary_artist"])

In [23]:
clean_data.to_parquet('../data/interim/tracks_enriched.parquet')
data.to_parquet('../data/interim/orig_data.parquet')